# M3 — Machine Learning

**Owner:** Vaishnavi  
**Input:** M2 `customer_features.parquet`  
**Constraint:** Temporal split; no prediction-period feature leakage.

See `docs/experiment_notes.md`.

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.modeling import (
    load_modelling_frame,
    validate_m2_handoff,
    assign_temporal_split,
    split_summary,
    train_and_evaluate,
    run_pipeline,
)

print("ROOT:", ROOT)

## 1. Load & validate M2 handoff

In [ ]:
df = load_modelling_frame()
checks = validate_m2_handoff(df)
print(json.dumps(checks, indent=2))
assert checks["all_passed"]
display(df.head())

## 2. Temporal split

In [ ]:
split = assign_temporal_split(df)
print(json.dumps(split_summary(df, split), indent=2))
display(split.value_counts())

## 3. Train LR / RF / XGBoost and select by validation PR-AUC

In [ ]:
results, pipelines, best = train_and_evaluate(df, split)
print("Selected:", best)
print(json.dumps(results["selection"], indent=2))
for name, block in results["models"].items():
    print(name, "val", {k: round(block["val"][k], 4) for k in ["roc_auc","pr_auc","precision","recall","f1"]})

## 4. Persist artifacts

In [ ]:
report = run_pipeline()
print("Final model:", report["artifacts"]["final_model"])
print(json.dumps(report["experiments"]["selection"]["selected_test_metrics"], indent=2))